In [0]:
import json
import logging
import uuid
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import (
    current_timestamp, lit, col, sha2, concat_ws, coalesce
)

#### Logger setup

Sets up a logger so each step prints a timestamped status line during the run.

In [0]:
logger = logging.getLogger("cdr_silver_transform")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger initialised.")

#### Parameters

Reads the same run_id the bronze task used, when this notebook runs as a job task after bronze.
Falls back to a new run_id when run on its own.

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_config_path", "/Volumes/cdr_bronze/config/table_config/table_config.json")
dbutils.widgets.text("bronze_catalog", "cdr_bronze")
dbutils.widgets.text("bronze_schema", "raw_mimic_iv")
dbutils.widgets.text("silver_catalog", "cdr_silver")
dbutils.widgets.text("silver_schema", "silver_mimic_iv")
dbutils.widgets.text("quarantine_schema", "quarantine_mimic_iv")

run_id_param = dbutils.widgets.get("run_id").strip()
table_config_path = dbutils.widgets.get("table_config_path")
bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
quarantine_schema = dbutils.widgets.get("quarantine_schema")

if run_id_param:
    run_id = run_id_param
else:
    try:
        run_id = dbutils.jobs.taskValues.get(taskKey="bronze_ingest", key="run_id", default="")
    except Exception:
        run_id = ""
    if not run_id:
        run_id = str(uuid.uuid4())

bronze_full_schema = f"{bronze_catalog}.{bronze_schema}"
silver_full_schema = f"{silver_catalog}.{silver_schema}"
quarantine_full_schema = f"{silver_catalog}.{quarantine_schema}"
audit_table = f"{silver_full_schema}.transformation_audit_log"

logger.info(f"run_id                 = {run_id}")
logger.info(f"bronze_full_schema     = {bronze_full_schema}")
logger.info(f"silver_full_schema     = {silver_full_schema}")
logger.info(f"quarantine_full_schema = {quarantine_full_schema}")
logger.info(f"audit_table            = {audit_table}")

#### Load table config

Reuses the same key config as bronze. The primary key drives both silver deduplication and the merge into the silver table.
A table set to null has no reliable key yet and falls back to a row hash.

In [0]:
with open(table_config_path, "r") as f:
    TABLE_CONFIG = json.load(f)

logger.info(f"TABLE_CONFIG loaded for {len(TABLE_CONFIG)} known tables from {table_config_path}.")

#### Schema setup

Creates the silver and quarantine schemas and the audit log table if they do not exist yet.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_full_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {quarantine_full_schema}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    run_id             STRING,
    source_table       STRING,
    target_table       STRING,
    load_type          STRING,
    status             STRING,
    rows_read          BIGINT,
    rows_passed        BIGINT,
    rows_quarantined   BIGINT,
    rows_inserted      BIGINT,
    rows_updated       BIGINT,
    start_time         TIMESTAMP,
    end_time           TIMESTAMP,
    duration_seconds   DOUBLE,
    notes              STRING,
    error_message      STRING
)
USING DELTA
""")

logger.info(f"Silver schema ready: {silver_full_schema}")
logger.info(f"Quarantine schema ready: {quarantine_full_schema}")
logger.info(f"Audit table ready: {audit_table}")

#### Write helper

One shared function for the merge-or-overwrite and audit logging mechanics, the same for every table.
Called once per table below, never in a loop over the table list.

In [0]:
audit_rows = []


def write_to_silver(df, clean_table_name, key_cols, quarantine_df=None, notes=None):
    target_table = f"{silver_full_schema}.{clean_table_name}"
    quarantine_table = f"{quarantine_full_schema}.{clean_table_name}"

    start_time = datetime.now()
    table_exists = spark.catalog.tableExists(target_table)
    load_type = "INCREMENTAL_MERGE" if table_exists else "INITIAL_LOAD"

    logger.info(f"Writing {target_table} [{load_type}]")

    rows_inserted = None
    rows_updated = None
    rows_quarantined = 0

    try:
        source_cols = df.columns

        df = (df
              .withColumn("_transformed_timestamp", current_timestamp())
              .withColumn("_run_id", lit(run_id)))

        if not key_cols:
            df = df.withColumn(
                "_row_key",
                sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("\u0000")) for c in source_cols]), 256)
            )
            merge_keys = ["_row_key"]
        else:
            merge_keys = key_cols

        rows_read = df.count()

        if load_type == "INITIAL_LOAD":
            (df.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(target_table))

            rows_inserted = rows_read
            rows_updated = 0

        else:  # INCREMENTAL_MERGE
            delta_target = DeltaTable.forName(spark, target_table)
            merge_condition = " AND ".join([f"t.{c} = s.{c}" for c in merge_keys])

            (delta_target.alias("t")
             .merge(df.alias("s"), merge_condition)
             .whenMatchedUpdateAll()
             .whenNotMatchedInsertAll()
             .execute())

            merge_metrics = (delta_target.history(1)
                              .select("operationMetrics")
                              .first()["operationMetrics"])
            rows_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
            rows_updated = int(merge_metrics.get("numTargetRowsUpdated", 0))

        if quarantine_df is not None and quarantine_df.count() > 0:
            (quarantine_df.write
             .format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(quarantine_table))
            rows_quarantined = quarantine_df.count()

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        logger.info(
            f"  SUCCESS | {rows_read} rows read | +{rows_inserted} inserted / "
            f"~{rows_updated} updated | {rows_quarantined} quarantined | "
            f"{duration:.1f}s -> {target_table}"
        )

        audit_rows.append(Row(
            run_id=run_id,
            source_table=f"{bronze_full_schema}.raw_{clean_table_name}",
            target_table=target_table,
            load_type=load_type,
            status="SUCCESS",
            rows_read=rows_read,
            rows_passed=rows_read - rows_quarantined,
            rows_quarantined=rows_quarantined,
            rows_inserted=rows_inserted,
            rows_updated=rows_updated,
            start_time=start_time,
            end_time=end_time,
            duration_seconds=duration,
            notes=notes,
            error_message=None
        ))

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_message = str(e)[:2000]

        logger.error(f"  FAILED | {target_table} | {error_message}")

        audit_rows.append(Row(
            run_id=run_id,
            source_table=f"{bronze_full_schema}.raw_{clean_table_name}",
            target_table=target_table,
            load_type=load_type,
            status="FAILED",
            rows_read=None,
            rows_passed=None,
            rows_quarantined=None,
            rows_inserted=None,
            rows_updated=None,
            start_time=start_time,
            end_time=end_time,
            duration_seconds=duration,
            notes=notes,
            error_message=error_message
        ))

#### Read all bronze tables

Each table is read individually into its own dictionary entry, no loop over the table list.

In [0]:
bronze_dfs = {}

bronze_dfs["admissions"] = spark.read.table(f"{bronze_full_schema}.raw_admissions")
bronze_dfs["caregiver"] = spark.read.table(f"{bronze_full_schema}.raw_caregiver")
bronze_dfs["chartevents"] = spark.read.table(f"{bronze_full_schema}.raw_chartevents")
bronze_dfs["d_hcpcs"] = spark.read.table(f"{bronze_full_schema}.raw_d_hcpcs")
bronze_dfs["d_icd_diagnoses"] = spark.read.table(f"{bronze_full_schema}.raw_d_icd_diagnoses")
bronze_dfs["d_icd_procedures"] = spark.read.table(f"{bronze_full_schema}.raw_d_icd_procedures")
bronze_dfs["d_items"] = spark.read.table(f"{bronze_full_schema}.raw_d_items")
bronze_dfs["d_labitems"] = spark.read.table(f"{bronze_full_schema}.raw_d_labitems")
bronze_dfs["datetimeevents"] = spark.read.table(f"{bronze_full_schema}.raw_datetimeevents")
bronze_dfs["demo_subject_id"] = spark.read.table(f"{bronze_full_schema}.raw_demo_subject_id")
bronze_dfs["diagnoses_icd"] = spark.read.table(f"{bronze_full_schema}.raw_diagnoses_icd")
bronze_dfs["drgcodes"] = spark.read.table(f"{bronze_full_schema}.raw_drgcodes")
bronze_dfs["emar"] = spark.read.table(f"{bronze_full_schema}.raw_emar")
bronze_dfs["emar_detail"] = spark.read.table(f"{bronze_full_schema}.raw_emar_detail")
bronze_dfs["hcpcsevents"] = spark.read.table(f"{bronze_full_schema}.raw_hcpcsevents")
bronze_dfs["icustays"] = spark.read.table(f"{bronze_full_schema}.raw_icustays")
bronze_dfs["ingredientevents"] = spark.read.table(f"{bronze_full_schema}.raw_ingredientevents")
bronze_dfs["inputevents"] = spark.read.table(f"{bronze_full_schema}.raw_inputevents")
bronze_dfs["labevents"] = spark.read.table(f"{bronze_full_schema}.raw_labevents")
bronze_dfs["microbiologyevents"] = spark.read.table(f"{bronze_full_schema}.raw_microbiologyevents")
bronze_dfs["omr"] = spark.read.table(f"{bronze_full_schema}.raw_omr")
bronze_dfs["outputevents"] = spark.read.table(f"{bronze_full_schema}.raw_outputevents")
bronze_dfs["patients"] = spark.read.table(f"{bronze_full_schema}.raw_patients")
bronze_dfs["pharmacy"] = spark.read.table(f"{bronze_full_schema}.raw_pharmacy")
bronze_dfs["poe"] = spark.read.table(f"{bronze_full_schema}.raw_poe")
bronze_dfs["poe_detail"] = spark.read.table(f"{bronze_full_schema}.raw_poe_detail")
bronze_dfs["prescriptions"] = spark.read.table(f"{bronze_full_schema}.raw_prescriptions")
bronze_dfs["procedureevents"] = spark.read.table(f"{bronze_full_schema}.raw_procedureevents")
bronze_dfs["procedures_icd"] = spark.read.table(f"{bronze_full_schema}.raw_procedures_icd")
bronze_dfs["provider"] = spark.read.table(f"{bronze_full_schema}.raw_provider")
bronze_dfs["services"] = spark.read.table(f"{bronze_full_schema}.raw_services")
bronze_dfs["transfers"] = spark.read.table(f"{bronze_full_schema}.raw_transfers")

logger.info(f"Read {len(bronze_dfs)} bronze table(s).")

#### Cleaning

Placeholder per table, validation and transformation rules go here next. For now every table passes through unchanged.

In [0]:
silver_dfs = {}

silver_dfs = {}

# d_labitems

KNOWN_LOWERCASE_PREFIXES = ("pH", "pCO2", "pO2", "pO2,", "pH,", "pCO2,")
KNOWN_MIXED_CASE_TOKENS = {"tacroFK", "eAG", "WB tCO2"}
AMBIGUOUS_FLUID_VALUES = ["Q", "I", "Fluid"]


def normalise_case(value):
    if value is None:
        return None

    value = str(value).strip()

    if value == "":
        return value

    if value.startswith(KNOWN_LOWERCASE_PREFIXES) or value in KNOWN_MIXED_CASE_TOKENS:
        return value

    if value.isupper() and len(value) <= 6:
        return value

    return value.title()


normalise_case_udf = F.udf(normalise_case, StringType())

silver_dfs["d_labitems"] = (
    bronze_dfs["d_labitems"]
    .withColumn("label", F.trim(F.col("label")))
    .withColumn("label", normalise_case_udf(F.col("label")))
    .withColumn(
        "label_missing",
        F.col("label").isNull() | (F.col("label") == "")
    )
    .withColumn("fluid", F.trim(F.col("fluid")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn(
        "fluid_flagged",
        F.col("fluid").isin(AMBIGUOUS_FLUID_VALUES)
    )
)


# d_hcpcs

silver_dfs["d_hcpcs"] = (
    bronze_dfs["d_hcpcs"]
    .select([
        F.trim(F.col(column_name)).alias(column_name)
        if data_type == "string"
        else F.col(column_name)
        for column_name, data_type in bronze_dfs["d_hcpcs"].dtypes
    ])
    .withColumn(
        "description",
        F.when(
            F.col("long_description").isNotNull()
            & (
                F.length(F.col("long_description"))
                >= F.length(F.col("short_description"))
            ),
            F.col("long_description")
        ).otherwise(F.col("short_description"))
    )
    .drop("long_description", "short_description")
    .dropDuplicates(["code", "category"])
)


# d_icd_diagnoses

d_icd_diagnoses_not_null = reduce(
    lambda left, right: left & right,
    [
        F.col(column_name).isNotNull()
        for column_name in bronze_dfs["d_icd_diagnoses"].columns
    ]
)

silver_dfs["d_icd_diagnoses"] = (
    bronze_dfs["d_icd_diagnoses"]
    .filter(d_icd_diagnoses_not_null)
    .select([
        F.trim(F.col(column_name)).alias(column_name)
        if data_type == "string"
        else F.col(column_name)
        for column_name, data_type in bronze_dfs["d_icd_diagnoses"].dtypes
    ])
    .dropDuplicates(["icd_code", "icd_version"])
)


# d_icd_procedures

silver_dfs["d_icd_procedures"] = (
    bronze_dfs["d_icd_procedures"]
    .withColumn(
        "icd_code",
        F.trim(F.col("icd_code").cast("string"))
    )
    .withColumn(
        "icd_version",
        F.col("icd_version").cast("int")
    )
)


# d_items

d_items_text_columns = [
    "label",
    "abbreviation",
    "linksto",
    "category",
    "unitname",
    "param_type"
]

silver_dfs["d_items"] = bronze_dfs["d_items"]

for column_name in d_items_text_columns:
    silver_dfs["d_items"] = silver_dfs["d_items"].withColumn(
        column_name,
        F.when(
            F.trim(F.col(column_name)) == "",
            None
        ).otherwise(
            F.trim(F.col(column_name))
        )
    )












#### Write all silver tables

Each table is written individually by calling the shared write_to_silver helper, no loop over the table list.

In [0]:
write_to_silver(silver_dfs["admissions"], "admissions", TABLE_CONFIG.get("admissions"))
write_to_silver(silver_dfs["caregiver"], "caregiver", TABLE_CONFIG.get("caregiver"))
write_to_silver(silver_dfs["chartevents"], "chartevents", TABLE_CONFIG.get("chartevents"))
write_to_silver(silver_dfs["d_hcpcs"], "d_hcpcs", TABLE_CONFIG.get("d_hcpcs"))
write_to_silver(silver_dfs["d_icd_diagnoses"], "d_icd_diagnoses", TABLE_CONFIG.get("d_icd_diagnoses"))
write_to_silver(silver_dfs["d_icd_procedures"], "d_icd_procedures", TABLE_CONFIG.get("d_icd_procedures"))
write_to_silver(silver_dfs["d_items"], "d_items", TABLE_CONFIG.get("d_items"))
write_to_silver(silver_dfs["d_labitems"], "d_labitems", TABLE_CONFIG.get("d_labitems"))
write_to_silver(silver_dfs["datetimeevents"], "datetimeevents", TABLE_CONFIG.get("datetimeevents"))
write_to_silver(silver_dfs["demo_subject_id"], "demo_subject_id", TABLE_CONFIG.get("demo_subject_id"))
write_to_silver(silver_dfs["diagnoses_icd"], "diagnoses_icd", TABLE_CONFIG.get("diagnoses_icd"))
write_to_silver(silver_dfs["drgcodes"], "drgcodes", TABLE_CONFIG.get("drgcodes"))
write_to_silver(silver_dfs["emar"], "emar", TABLE_CONFIG.get("emar"))
write_to_silver(silver_dfs["emar_detail"], "emar_detail", TABLE_CONFIG.get("emar_detail"))
write_to_silver(silver_dfs["hcpcsevents"], "hcpcsevents", TABLE_CONFIG.get("hcpcsevents"))
write_to_silver(silver_dfs["icustays"], "icustays", TABLE_CONFIG.get("icustays"))
write_to_silver(silver_dfs["ingredientevents"], "ingredientevents", TABLE_CONFIG.get("ingredientevents"))
write_to_silver(silver_dfs["inputevents"], "inputevents", TABLE_CONFIG.get("inputevents"))
write_to_silver(silver_dfs["labevents"], "labevents", TABLE_CONFIG.get("labevents"))
write_to_silver(silver_dfs["microbiologyevents"], "microbiologyevents", TABLE_CONFIG.get("microbiologyevents"))
write_to_silver(silver_dfs["omr"], "omr", TABLE_CONFIG.get("omr"))
write_to_silver(silver_dfs["outputevents"], "outputevents", TABLE_CONFIG.get("outputevents"))
write_to_silver(silver_dfs["patients"], "patients", TABLE_CONFIG.get("patients"))
write_to_silver(silver_dfs["pharmacy"], "pharmacy", TABLE_CONFIG.get("pharmacy"))
write_to_silver(silver_dfs["poe"], "poe", TABLE_CONFIG.get("poe"))
write_to_silver(silver_dfs["poe_detail"], "poe_detail", TABLE_CONFIG.get("poe_detail"))
write_to_silver(silver_dfs["prescriptions"], "prescriptions", TABLE_CONFIG.get("prescriptions"))
write_to_silver(silver_dfs["procedureevents"], "procedureevents", TABLE_CONFIG.get("procedureevents"))
write_to_silver(silver_dfs["procedures_icd"], "procedures_icd", TABLE_CONFIG.get("procedures_icd"))
write_to_silver(silver_dfs["provider"], "provider", TABLE_CONFIG.get("provider"))
write_to_silver(silver_dfs["services"], "services", TABLE_CONFIG.get("services"))
write_to_silver(silver_dfs["transfers"], "transfers", TABLE_CONFIG.get("transfers"))

#### Update the audit table

Writes all audit rows from this run to the audit table and logs a final summary.

In [0]:
success_count = sum(1 for r in audit_rows if r["status"] == "SUCCESS")
failure_count = sum(1 for r in audit_rows if r["status"] == "FAILED")

if audit_rows:
    audit_df = spark.createDataFrame(audit_rows)
    audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
    logger.info(f"Audit log written: {len(audit_rows)} row(s) -> {audit_table}")

logger.info(
    f"Run {run_id} complete | {success_count} succeeded | {failure_count} failed "
    f"| {len(audit_rows)} total tables"
)

if failure_count > 0:
    logger.warning(f"{failure_count} table(s) failed, check {audit_table} for details (run_id = {run_id})")